# Scraping Komentar TikTok (Profil & URL)
Notebook ini digunakan untuk melakukan scraping data komentar TikTok menggunakan layanan cloud Apify (`apify_client`).

In [5]:
import os
import pandas as pd
from tqdm import tqdm
from apify_client import ApifyClient
from datetime import datetime

## 1. Konfigurasi Apify Token & Target
Dapatkan API token Anda dari dashboard Apify (Settings -> Integrations).

In [ ]:
# Masukkan Apify API Token Anda
APIFY_TOKEN = "your-apify-token"

# ==================================================================
# CARA MENGGUNAKAN SATU METODE SAJA:
# - Jika hanya ingin memakai Profil: Kosongkan list target_tiktok_urls = []
# - Jika hanya ingin memakai URL Spesifik: Kosongkan list target_tiktok_accounts = []
# ==================================================================

# Target 1: Berdasarkan Username/Profil (akan mengambil video-video terbaru)
target_tiktok_accounts = [
    # "@bpjskesehatan_ri",
    # "@ditjenpajakri", 
    "@badangizinasional.ri"
]

# Target 2: Berdasarkan URL Video Spesifik
target_tiktok_urls = [
    # Coretax
    # "https://vt.tiktok.com/ZSxLGJqUK/",
    # "https://vt.tiktok.com/ZSxLscFaw/",
    # "https://vt.tiktok.com/ZSxLGA1PF/",
    # "https://vt.tiktok.com/ZSxLGhHuq/",
    # "https://vt.tiktok.com/ZSxLGkpS5/",
    # "https://vt.tiktok.com/ZSxLGUfNQ/",
    # "https://vt.tiktok.com/ZSxLGV68g/",
    # "https://vt.tiktok.com/ZSxLGkHjS/",
    # "https://vt.tiktok.com/ZSxLtePb6/",
    # "https://vt.tiktok.com/ZSxLtJ9K4/",
    # "https://vt.tiktok.com/ZSxLG7wGF/",
    # "https://vt.tiktok.com/ZSxLGncc9/",
    # "https://vt.tiktok.com/ZSxLGb4fn/",
    # "https://vt.tiktok.com/ZSxLGpAB7/",
    # "https://vt.tiktok.com/ZSxLGgR42/",
    # "https://vt.tiktok.com/ZSxLt2uea/",
    
    # MBG
    # "https://vt.tiktok.com/ZSxLtSMHr/",
    # "https://vt.tiktok.com/ZSxLtk94X/",
    # "https://vt.tiktok.com/ZSxLtyGLJ/",
    # "https://vt.tiktok.com/ZSxLtBtem/",
    # "https://vt.tiktok.com/ZSxLtfRCg/",
    # "https://vt.tiktok.com/ZSxNyR2T1/",

    # BPJS Kesehatan
    # "https://vt.tiktok.com/ZSxLtsvgv/",
    # "https://vt.tiktok.com/ZSxLtsUvq/",
    # "https://vt.tiktok.com/ZSxLtsyUE/",
    # "https://vt.tiktok.com/ZSxLtoNuV/",
    # "https://vt.tiktok.com/ZSxLtEoDX/",
    # "https://vt.tiktok.com/ZSxLtpjuk/",
    # "https://vt.tiktok.com/ZSxLtKfhN/",
    # "https://vt.tiktok.com/ZSxLn1S2J/",

]

# Jumlah maksimal komentar yang ingin diambil per postingan/video
MAX_COMMENTS_PER_POST = 0

# Batas pengaman saldo Apify (Maksimal ~4.000 untuk limit $5)
MAX_TOTAL_ITEMS = 6000

## 2. Fungsi Scraping Menggunakan Apify Actor
Fungsi ini memicu actor TikTok Scraper (`BDec00yAmCm1QbMEI`) di Apify untuk melakukan scraping.

In [19]:
def scrape_tiktok_comments(api_token, profiles, urls, comments_per_post=99999, max_items=4000):
    client = ApifyClient(api_token)
    
    run_input = {
        "commentsPerPost": 99999,      
        "excludePinnedPosts": False,
        "maxRepliesPerComment": 0,   
        "profileScrapeSections": ["videos"],
        "profileSorting": "latest",  
        "profiles": profiles,
        "postURLs": urls,
        "resultsPerPage": 100,
        "maxItems": max_items  # Proteksi batas limit saldo
    }
    
    print(">>> Memulai pekerjaan scraping di server Apify...")
    try:
        run = client.actor("BDec00yAmCm1QbMEI").call(run_input=run_input)
        print("Pekerjaan scraping berhasil diselesaikan di Apify!\n")
    except Exception as e:
        print(f"Gagal memanggil Apify Actor: {e}")
        return pd.DataFrame()
        
    results = []
    dataset_id = run["defaultDatasetId"]
    
    for item in tqdm(client.dataset(dataset_id).iterate_items(), desc="Mengunduh data TikTok"):
        results.append({
            "id": str(item.get("cid")),
            "timestamp": item.get("createTimeISO"),
            "likesCount": item.get("diggCount", 0),
            "postUrl": item.get("videoWebUrl"),
            "commentUrl": f"{item.get('videoWebUrl')}?commentId={item.get('cid')}",
            "source_file": "scraping_tiktok",
            "ownerUsername": item.get("uniqueId") or item.get("user", {}).get("uniqueId"),
            "text": item.get("text")
        })
        
    return pd.DataFrame(results)

## 3. Jalankan Scraping & Simpan ke CSV
Format CSV akan disesuaikan dengan format dataset pelabelan Anda.

In [20]:
df_tiktok = scrape_tiktok_comments(APIFY_TOKEN, target_tiktok_accounts, target_tiktok_urls, 
                                   comments_per_post=MAX_COMMENTS_PER_POST,
                                   max_items=MAX_TOTAL_ITEMS)

if not df_tiktok.empty:
    df_tiktok = df_tiktok.dropna(subset=['text'])
    
    output_dir = r"C:\Users\Lenovo\Downloads\skrips_code\xgboost_method\hasil_scraping\tiktok"
    os.makedirs(output_dir, exist_ok=True)
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_filename = os.path.join(output_dir, f"dataset_tiktok_{timestamp}.csv")
    
    df_tiktok.to_csv(output_filename, index=False, encoding='utf-8-sig')
    
    print("\n" + "="*50)
    print("✅ SCRAPING TIKTOK SELESAI!")
    print(f"Total Komentar Terambil: {len(df_tiktok)}")
    print(f"Data berhasil disimpan ke:\n{output_filename}")
    print("="*50)
else:
    print("\n❌ Scraping gagal atau tidak ada komentar yang didapatkan.")

>>> Memulai pekerjaan scraping di server Apify...


[apify.tiktok-comments-scraper runId:q1pD1pHcspODdGQwg] -> Status: RUNNING, Message: 
[apify.tiktok-comments-scraper runId:q1pD1pHcspODdGQwg] -> 2026-05-25T12:05:46.801Z ACTOR: Pulling container image of build 0DdaaVYiwbx3balcj from registry.
[apify.tiktok-comments-scraper runId:q1pD1pHcspODdGQwg] -> 2026-05-25T12:05:46.803Z ACTOR: Creating container.
[apify.tiktok-comments-scraper runId:q1pD1pHcspODdGQwg] -> 2026-05-25T12:05:46.854Z ACTOR: Starting container.
[apify.tiktok-comments-scraper runId:q1pD1pHcspODdGQwg] -> 2026-05-25T12:05:46.855Z ACTOR: Running under "LIMITED_PERMISSIONS".
[apify.tiktok-comments-scraper runId:q1pD1pHcspODdGQwg] -> 2026-05-25T12:05:47.051Z Running on architecture: x86_64
[apify.tiktok-comments-scraper runId:q1pD1pHcspODdGQwg] -> 2026-05-25T12:05:47.051Z Will run command: xvfb-run -a -s "-ac -screen 0 1920x1080x24+32 -nolisten tcp" /bin/bash -o pipefail -c bash -c "    java         --enable-native-access=ALL-UNNAMED         -cp classes-test:classes-main:lib/

Pekerjaan scraping berhasil diselesaikan di Apify!



Mengunduh data TikTok: 4000it [00:09, 432.81it/s]


✅ SCRAPING TIKTOK SELESAI!
Total Komentar Terambil: 4000
Data berhasil disimpan ke:
C:\Users\Lenovo\Downloads\skrips_code\xgboost_method\hasil_scraping\tiktok\dataset_tiktok_20260525_201049.csv
